In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

In [12]:
df = pd.read_csv("../data/processed_reviews.csv")

y = df["sentiment"]

tf = TfidfVectorizer(max_features=10000)
X = tf.fit_transform(df["review"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [13]:
X_train = X_train.toarray()
X_test = X_test.toarray()

train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

train_loader = DataLoader(
    train_set,
    shuffle=True,
    batch_size=64
)

test_loader = DataLoader(
    test_set,
    shuffle=False,
    batch_size=64
)

In [14]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN Layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # Fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        
        output, _ = self.rnn(x, h0)

        output = self.fc(output[:, -1, :])
        return output

In [15]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

### Training the RNN

In [17]:
epochs = 20

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        outputs = torch.sigmoid(outputs.squeeze())

        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    print(f"epoch = {epoch + 1}/{epochs} and loss = {loss.item()}")

epoch = 1/20 and loss = 0.17468750476837158
epoch = 2/20 and loss = 0.11731036007404327
epoch = 3/20 and loss = 0.247323676943779
epoch = 4/20 and loss = 0.06436726450920105
epoch = 5/20 and loss = 0.15915650129318237
epoch = 6/20 and loss = 0.47219786047935486
epoch = 7/20 and loss = 0.11858773976564407
epoch = 8/20 and loss = 0.15497323870658875
epoch = 9/20 and loss = 0.09086180478334427
epoch = 10/20 and loss = 0.11022688448429108
epoch = 11/20 and loss = 0.05871050804853439
epoch = 12/20 and loss = 0.03932435065507889
epoch = 13/20 and loss = 0.09240637719631195
epoch = 14/20 and loss = 0.17185091972351074
epoch = 15/20 and loss = 0.049058783799409866
epoch = 16/20 and loss = 0.07470957934856415
epoch = 17/20 and loss = 0.04894419014453888
epoch = 18/20 and loss = 0.07891110330820084
epoch = 19/20 and loss = 0.05575370788574219
epoch = 20/20 and loss = 0.011888742446899414


 ### Evaluation

In [18]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals / tot_vals * 100}")

accuracy = 84.20893415347383
